# EDA: Engineering Tools Discovery Assistant (ДЗ №2)

**Цель:** оценить состав, объём и качество корпуса для поиска/ранжирования инженерных инструментов. 

**Данные:** 18 реальных open-source инструментов из публичных источников; 51 запрос, сформулированный по типовым инженерным задачам; 918 пар с релевантностью по матрице соответствия. В карточках использованы авторские описания со ссылками на проекты. Открытый корпус воспроизводится полностью; отдельно подготовленные 5 закрытых инструментов в EDA не входят. Методика - `../data/README.md`.


In [2]:
import sys
from pathlib import Path

ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))
from scripts.eda import make_charts  # noqa: E402

eda = make_charts()
tools, queries, labels = eda["tools"], eda["queries"], eda["judgments"]
print(f"Инструментов: {len(tools)} | Запросов: {len(queries)} | Пар: {len(labels)}")
print("Проверки данных:", eda["quality"])

Инструментов: 18 | Запросов: 51 | Пар: 918
Проверки данных: {'missing_cells': 0, 'duplicate_tool_ids': 0, 'duplicate_query_texts': 0, 'duplicate_pairs': 0}


## 1. Состав и покрытие каталога

Каждый `tool_id` ссылается на реальный проект (`source_url`), но его поля описания составлены вручную. Шесть категорий позволяют проверить разнообразие, однако по 2-4 инструментам нельзя судить о полноценном покрытии каждой области.

![Инструменты по категориям](../reports/figures/01_tool_categories.png)


In [3]:
print(tools.groupby("category").size().to_string())
print("Уникальные ссылки:", tools.source_url.nunique())

category
debugging      4
http           3
monitoring     3
network        3
performance    2
storage        3
Уникальные ссылки: 18


## 2. Качество и длина текста

Структурные ошибки в критичных полях отсутствуют. Длина карточек мала: это краткие карточки назначения инструментов, а не полнотекстовые README. Длинные технические вопросы могут содержать термины, которых нет в карточке, поэтому надо проверить работу с расширенными описаниями и двуязычным представлением.

![Длина описаний](../reports/figures/02_description_lengths.png)


In [4]:
print("Медиана слов в RU описании:", tools.ru_words.median())
print("Минимум / максимум:", tools.ru_words.min(), tools.ru_words.max())
print("Пропуски в критичных столбцах:", eda["quality"]["missing_cells"])

Медиана слов в RU описании: 16.0
Минимум / максимум: 13 19
Пропуски в критичных столбцах: 0


## 3. Состав запросов и схема разделения

17 семейств намерений, в каждом три авторские перефразировки. Все перефразировки относятся к **одному** split. Train/validation/test представлены 33/9/9 запросами, но распределение тематик неоднородно. Тест включает три запроса вне каталога; для оценки ранжирования остаётся только шесть answerable-запросов. Такая оценка будет шумной.

![Запросы по категориям](../reports/figures/03_query_coverage.png)


In [5]:
print(queries.groupby(["split", "category"]).size().to_string())
print("Семейств с утечкой по split:", (queries.groupby("family_id").split.nunique() > 1).sum())

split       category   
test        network        3
            other          3
            storage        3
train       debugging      6
            http           6
            monitoring     6
            network        6
            performance    6
            storage        3
validation  debugging      3
            monitoring     3
            storage        3
Семейств с утечкой по split: 0


## 4. Качество разметки, а не только баланс классов

Релевантность: 2 - прямое соответствие, 1 - частичное, 0 - не вошёл в список релевантных. **Нули автоматически заполнены и по отдельности не проверены специалистом**, поэтому формальная доля класса 0 не оценивает истинную нерелевантность. Прежде чем обучать модель и оценивать её на тесте, нужно пересмотреть 162 пары test, а затем выборочно train/validation.

![Распределение меток](../reports/figures/04_relevance_labels.png)


In [6]:
print(labels.groupby(["relevance", "annotation_method"]).size().to_string())
print("Число пар:", len(labels))

relevance  annotation_method          
0          not_in_capability_mapping      846
1          capability_mapping_positive     24
2          capability_mapping_positive     48
Число пар: 918


## 5. Релевантные инструменты на запрос и no-answer

На некоторых запросах уместны две утилиты, но есть и задача вне области каталога. Поэтому для запросов с ответом использовать Recall@3 / nDCG@5; для no-answer - отдельную проверку отказа, не подменяя её нулевым nDCG.

![Релевантных инструментов на запрос](../reports/figures/05_relevance_per_query.png)


In [7]:
print(eda["positives"].groupby(["split", "relevant"]).size().to_string())
print("Запросов без релевантной утилиты:", int(eda["positives"].relevant.eq(0).sum()))

split       relevant
test        0            3
            1            3
            2            3
train       1           21
            2           12
validation  2            9
Запросов без релевантной утилиты: 3


## 6. Длина запросов и языковой риск

Короткие русскоязычные запросы содержат англоязычные термины (IOPS, TCP, NVMe). Это повод проверить простую токенизацию BM25 и двуязычные карточки, а не предполагать, что сложный reranker обязательно лучше.

![Длины запросов](../reports/figures/06_query_lengths.png)


In [8]:
print(
    "Длина запроса: медиана / min / max:",
    queries.words.median(),
    queries.words.min(),
    queries.words.max(),
)

Длина запроса: медиана / min / max: 8.0 4 11
